# Proyecto Productivo IB · Semana 11
## Desarrollo del componente analítico · Equipo AgroData IC (Grupo 03)  
**Proyecto:** alerta temprana por grados-día para la arañita roja (*Oligonychus punicae*) en palto Hass · Fundo Loma Verde S.A.C.  
**Entrada:** `aranita_roja_analitico_v1.csv` (Semana 9). **Producto:** componente analítico funcional + ficha técnica + justificación de la técnica.

**El componente:** un **semáforo de infestación por módulo** (Verde / Ámbar / Rojo / Datos insuficientes) basado en una **regla determinística configurable** sobre el % de hojas infestadas observado en campo. El GDA acumulado se muestra como **contexto térmico**, no como disparador, porque en la Semana 10 su relación con la infestación resultó débil (ρ = 0.25, p = 0.08).

> Los umbrales de 10 % y 30 % son **provisionales** (corresponden aproximadamente a los percentiles 68 y 90 de nuestros datos). Deben validarse con el especialista de campo.

## 0 · Parámetros configurables

In [ ]:
import os, shutil
import numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display

# --- PARÁMETROS DEL COMPONENTE (provisionales, a validar con el especialista) ---
UMBRAL_AMBAR = 10.0       # % de hojas infestadas desde el cual se pasa a Ámbar
UMBRAL_ROJO  = 30.0       # % de hojas infestadas desde el cual se pasa a Rojo
VENTANA_DIAS = 14         # días hacia atrás desde la fecha de corte para el semáforo por módulo
MIN_EVAL     = 5          # mínimo de evaluaciones en la ventana; menos → 'Datos insuficientes'
FECHA_CORTE  = '2026-08-28'  # última fecha con evaluación en el dataset
TBASE, K = 10.34, 163.93     # supuestos de literatura (solo para el contexto térmico)

pd.set_option('display.width', 180); pd.set_option('display.max_columns', 40)
SALIDA = Path('salidas_semana11'); SALIDA.mkdir(exist_ok=True)
print('Umbrales:', UMBRAL_AMBAR, '/', UMBRAL_ROJO, '| ventana:', VENTANA_DIAS, 'días | mínimo de evaluaciones:', MIN_EVAL)

## PASO 1 · Cargar el CSV real de la Semana 9
Sube `aranita_roja_analitico_v1.csv`. No se generan datos de demostración.

In [ ]:
ARCHIVO = 'aranita_roja_analitico_v1.csv'
if not os.path.exists(ARCHIVO):
    try:
        from google.colab import files
        up = files.upload()
        if not up: raise ValueError('Debes seleccionar el CSV analítico de la Semana 9')
        nombre = list(up.keys())[0]
        if nombre != ARCHIVO: os.rename(nombre, ARCHIVO)
    except ImportError:
        raise FileNotFoundError(f'Coloca {ARCHIVO} junto al notebook.')
df = pd.read_csv(ARCHIVO, encoding='utf-8-sig')
fuente = ARCHIVO
print('Fuente:', fuente, '| dimensiones:', df.shape)
display(df.head(3))

## PASO 2 · Diseño de la regla (antes de programar)
| Elemento | Definición |
|---|---|
| **Decisión que apoya** | ¿Qué módulos requieren revisión o intervención esta semana por arañita roja? |
| **Entradas** | `evaluacion_id`, `fecha`, `modulo`, `hojas_inf_pct`; parámetros `UMBRAL_AMBAR`, `UMBRAL_ROJO`, `VENTANA_DIAS`, `MIN_EVAL`, `FECHA_CORTE` |
| **Validación** | `evaluacion_id` presente; `fecha` válida; `modulo` presente; `hojas_inf_pct` numérico, finito y entre 0 y 100. Lo que no cumple es **No evaluable** (no se vuelve 0 ni se elimina) |
| **Regla por evaluación** | `< UMBRAL_AMBAR` → Verde · `≥ UMBRAL_AMBAR` y `< UMBRAL_ROJO` → Ámbar · `≥ UMBRAL_ROJO` → Rojo |
| **Semáforo por módulo** | Promedio de `hojas_inf_pct` de las evaluaciones **evaluables** de los últimos `VENTANA_DIAS` días hasta `FECHA_CORTE`, clasificado con los mismos umbrales. Con menos de `MIN_EVAL` evaluables → Datos insuficientes. Además se reporta el máximo, el número de evaluaciones en Rojo y una observación de “focos” si el promedio lo oculta |
| **Salidas** | Estado por evaluación, indicadores globales, semáforo por módulo, tabla de revisión |

## PASO 3 · Componente (funciones reutilizables)
El componente queda en funciones para poder probarlo con datos limpios, sucios y vacíos.

In [ ]:
ESTADOS = ['Verde', 'Ámbar', 'Rojo']

def validar_entradas(datos):
    '''Marca cada fila como evaluable o no, con el motivo. No elimina ni convierte a 0 los inválidos.'''
    requeridas = ['evaluacion_id', 'fecha', 'modulo', 'hojas_inf_pct']
    faltantes = [c for c in requeridas if c not in datos.columns]
    if faltantes:
        raise ValueError(f'Faltan columnas obligatorias: {faltantes}. Disponibles: {list(datos.columns)}')
    t = datos.copy()
    t['hojas_inf_pct'] = pd.to_numeric(t['hojas_inf_pct'], errors='coerce')
    t.loc[~np.isfinite(t['hojas_inf_pct']), 'hojas_inf_pct'] = np.nan      # inf y NaN → no válido
    t['fecha'] = pd.to_datetime(t['fecha'], errors='coerce')
    id_ok = t['evaluacion_id'].notna() & t['evaluacion_id'].astype('string').str.strip().ne('').fillna(False)
    fecha_ok = t['fecha'].notna()
    modulo_ok = t['modulo'].notna()
    valor_ok = t['hojas_inf_pct'].notna() & t['hojas_inf_pct'].between(0, 100)
    t['evaluable'] = (id_ok & fecha_ok & modulo_ok & valor_ok).fillna(False)
    t['motivo'] = 'Válido'
    t.loc[~valor_ok, 'motivo'] = 'Infestación inválida'
    t.loc[~modulo_ok, 'motivo'] = 'Módulo ausente'
    t.loc[~fecha_ok, 'motivo'] = 'Fecha inválida'
    t.loc[~id_ok, 'motivo'] = 'ID inválido'
    return t

def clasificar(valor, umbral_ambar=UMBRAL_AMBAR, umbral_rojo=UMBRAL_ROJO):
    '''Regla determinística para un valor de % de hojas infestadas.'''
    if not (np.isfinite(umbral_ambar) and np.isfinite(umbral_rojo)) or not (0 <= umbral_ambar < umbral_rojo <= 100):
        raise ValueError('Los umbrales deben cumplir 0 <= ámbar < rojo <= 100')
    try:
        v = float(valor)
    except (TypeError, ValueError):
        return 'No evaluable'
    if not np.isfinite(v) or v < 0 or v > 100:
        return 'No evaluable'
    if v >= umbral_rojo:
        return 'Rojo'
    if v >= umbral_ambar:
        return 'Ámbar'
    return 'Verde'

def aplicar_regla(t, umbral_ambar=UMBRAL_AMBAR, umbral_rojo=UMBRAL_ROJO):
    r = t.copy()
    r['estado'] = [clasificar(v, umbral_ambar, umbral_rojo) if ev else 'No evaluable' for v, ev in zip(r['hojas_inf_pct'], r['evaluable'])]
    return r

def calcular_indicadores(r, umbral_ambar=UMBRAL_AMBAR, umbral_rojo=UMBRAL_ROJO):
    '''El denominador de los porcentajes son SOLO las evaluaciones evaluables.'''
    total = len(r); evaluables = int(r['evaluable'].sum())
    cuentas = {e: int((r['estado'] == e).sum()) for e in ESTADOS}
    ind = {'Total': total, 'Evaluables': evaluables, 'No_evaluables': total - evaluables,
           'Verde': cuentas['Verde'], 'Ambar': cuentas['Ámbar'], 'Rojo': cuentas['Rojo'],
           'Pct_Ambar_o_Rojo': 100 * (cuentas['Ámbar'] + cuentas['Rojo']) / evaluables if evaluables else np.nan,
           'Pct_Rojo': 100 * cuentas['Rojo'] / evaluables if evaluables else np.nan,
           'Umbral_Ambar': umbral_ambar, 'Umbral_Rojo': umbral_rojo}
    assert ind['Evaluables'] + ind['No_evaluables'] == total
    assert ind['Verde'] + ind['Ambar'] + ind['Rojo'] == evaluables
    return pd.DataFrame([ind])

def semaforo_modulos(r, fecha_corte=FECHA_CORTE, ventana_dias=VENTANA_DIAS, min_eval=MIN_EVAL,
                     umbral_ambar=UMBRAL_AMBAR, umbral_rojo=UMBRAL_ROJO):
    '''Semáforo por módulo con las evaluaciones evaluables de los últimos `ventana_dias` días hasta la fecha de corte.'''
    corte = pd.Timestamp(fecha_corte); inicio = corte - pd.Timedelta(days=ventana_dias - 1)
    ev = r[r['evaluable'] & (r['fecha'] >= inicio) & (r['fecha'] <= corte)]
    filas = []
    for m in sorted(r['modulo'].dropna().unique()):
        sub = ev[ev['modulo'] == m]
        if len(sub) < min_eval:
            estado, media, maximo = 'Datos insuficientes', (sub['hojas_inf_pct'].mean() if len(sub) else np.nan), (sub['hojas_inf_pct'].max() if len(sub) else np.nan)
        else:
            media, maximo = sub['hojas_inf_pct'].mean(), sub['hojas_inf_pct'].max()
            estado = clasificar(media, umbral_ambar, umbral_rojo)
        n_rojas = int((sub['hojas_inf_pct'] >= umbral_rojo).sum())
        obs = 'Revisar focos: hay evaluaciones en Rojo aunque el promedio es menor' if (estado in ('Verde', 'Ámbar') and n_rojas > 0) else ''
        filas.append({'modulo': m, 'evaluaciones_en_ventana': len(sub), 'fechas_en_ventana': sub['fecha'].nunique(),
                      'hojas_inf_media': media, 'hojas_inf_maxima': maximo, 'evaluaciones_rojas': n_rojas, 'estado_modulo': estado, 'observacion': obs,
                      'ventana_inicio': inicio.date(), 'fecha_corte': corte.date()})
    return pd.DataFrame(filas)
print('Funciones definidas: validar_entradas, clasificar, aplicar_regla, calcular_indicadores, semaforo_modulos')

## PASO 4 · Validar entradas y aplicar la regla

In [ ]:
trabajo = validar_entradas(df)
print('Evaluables:', int(trabajo['evaluable'].sum()), '| No evaluables:', int((~trabajo['evaluable']).sum()))
print(trabajo['motivo'].value_counts().to_string())
resultado = aplicar_regla(trabajo)
display(resultado['estado'].value_counts().rename('evaluaciones').to_frame())
display(resultado[['evaluacion_id', 'fecha', 'modulo', 'hojas_inf_pct', 'evaluable', 'estado']].head(8))

## PASO 5 · Indicadores globales
El denominador de los porcentajes es **solo evaluables**. Si no hay evaluables, el porcentaje es no calculable (`NaN`).

In [ ]:
indicadores = calcular_indicadores(resultado)
indicadores['Fuente'] = fuente
display(indicadores.round(2))
assert int(indicadores.loc[0, 'Total']) == 1462 and int(indicadores.loc[0, 'Evaluables']) == 1462

## PASO 6 · Semáforo por módulo en la ventana más reciente

In [ ]:
semaforo = semaforo_modulos(resultado)
display(semaforo.round(2))
print('Ventana:', semaforo.loc[0, 'ventana_inicio'], 'a', semaforo.loc[0, 'fecha_corte'])

> **Cómo leerlo:** el promedio puede esconder focos (Hallazgo 1 de la Semana 10), por eso la tabla muestra también la infestación máxima, cuántas evaluaciones están en Rojo y una **observación** cuando un módulo sale Verde o Ámbar con focos en Rojo. “Datos insuficientes” **no** significa sano; significa que no hay base para opinar. Un módulo Verde con pocas evaluaciones tampoco es garantía: por eso se muestra también la infestación máxima de la ventana.

## PASO 7 · Tabla de revisión
Evaluaciones Ámbar o Rojo dentro de la ventana, ordenadas de mayor a menor infestación. Es la lista que un técnico revisaría primero.

In [ ]:
inicio = pd.Timestamp(FECHA_CORTE) - pd.Timedelta(days=VENTANA_DIAS - 1)
revision = (resultado[resultado['estado'].isin(['Ámbar', 'Rojo']) & (resultado['fecha'] >= inicio) & (resultado['fecha'] <= pd.Timestamp(FECHA_CORTE))]
            .sort_values(['hojas_inf_pct', 'fecha'], ascending=[False, False])
            [['evaluacion_id', 'fecha', 'modulo', 'turno', 'valvula', 'hojas_inf_pct', 'estado']].copy())
print('Evaluaciones Ámbar/Rojo en la ventana:', len(revision))
display(revision.head(10))

## PASO 8 · Contexto térmico (informativo)
El GDA acumulado **no** interviene en el semáforo. Se reporta como contexto: cuántas generaciones teóricas de la plaga acumula la campaña con los supuestos de literatura (Tbase = 10.34 °C, K = 163.93 GDA).

In [ ]:
ultima = df[df['fecha'] == FECHA_CORTE].iloc[0]
contexto = pd.DataFrame([{'fecha_corte': FECHA_CORTE, 'gda_acum': round(ultima['gda_acum'], 1), 'generaciones_acum': round(ultima['gda_acum'] / K, 2),
                          'Tbase': TBASE, 'K': K,
                          'nota': 'Supuestos de literatura no validados en campo; el GDA no activa el semáforo.'}])
display(contexto)
assert abs(contexto.loc[0, 'generaciones_acum'] - 12.29) < 0.01

## PASO 9 · Pruebas del componente
Casos normales, **casos límite** (exactamente en el umbral), valores inválidos, parámetros configurables, datos sucios y vacíos.

In [ ]:
# 1-4 · Casos normales
assert clasificar(0) == 'Verde'
assert clasificar(5.5) == 'Verde'
assert clasificar(20) == 'Ámbar'
assert clasificar(65) == 'Rojo'
# 5-8 · Casos límite: justo en y justo debajo del umbral
assert clasificar(9.99) == 'Verde'
assert clasificar(10) == 'Ámbar'          # el umbral pertenece al estado superior
assert clasificar(29.99) == 'Ámbar'
assert clasificar(30) == 'Rojo'
# 9-10 · Extremos válidos
assert clasificar(100) == 'Rojo'
assert clasificar(0.0) == 'Verde'
# 11-15 · Inválidos → No evaluable (no se vuelven 0)
for malo in [-5, 101, float('nan'), float('inf'), 'abc', None]:
    assert clasificar(malo) == 'No evaluable', malo
# 16 · Umbral configurable
assert clasificar(15, umbral_ambar=20, umbral_rojo=40) == 'Verde'
assert clasificar(15, umbral_ambar=5, umbral_rojo=12) == 'Rojo'
# 17 · Parámetros incoherentes se rechazan
for a, r_ in [(30, 10), (10, 10), (-1, 30), (10, 150), (float('nan'), 30)]:
    try:
        clasificar(5, a, r_); raise AssertionError('debió fallar con umbrales incoherentes')
    except ValueError:
        pass
print('✓ Pruebas de la regla superadas')

In [ ]:
# 18-22 · Fixture sucio: el flujo completo con datos inválidos
sucio = pd.DataFrame({
  'evaluacion_id': [1, 2, 3, None, 5, 6, 7, 8],
  'fecha': ['2026-08-27', '2026-08-27', '2026-08-26', '2026-08-26', 'no-es-fecha', '2026-08-25', '2026-08-20', '2026-08-28'],
  'modulo': [4, 4, 4, 4, 4, 4, 5, None],
  'hojas_inf_pct': [5, 15, 'abc', 40, 50, -3, 45, 10]})
ts = aplicar_regla(validar_entradas(sucio))
esperado = ['Verde', 'Ámbar', 'No evaluable', 'No evaluable', 'No evaluable', 'No evaluable', 'Rojo', 'No evaluable']
assert ts['estado'].tolist() == esperado, ts['estado'].tolist()
assert len(ts) == len(sucio)                                       # no se eliminó ninguna fila
assert ts.loc[2, 'hojas_inf_pct'] != 0 and np.isnan(ts.loc[2, 'hojas_inf_pct'])   # el inválido NO se convirtió en 0
assert ts['motivo'].tolist() == ['Válido', 'Válido', 'Infestación inválida', 'ID inválido', 'Fecha inválida', 'Infestación inválida', 'Válido', 'Módulo ausente']
ind = calcular_indicadores(ts)
assert ind.loc[0, 'Total'] == 8 and ind.loc[0, 'Evaluables'] == 3 and ind.loc[0, 'No_evaluables'] == 5
assert abs(ind.loc[0, 'Pct_Ambar_o_Rojo'] - 100 * 2 / 3) < 1e-9     # denominador = solo evaluables (3), no el total (8)
assert ind.loc[0, 'Verde'] + ind.loc[0, 'Ambar'] + ind.loc[0, 'Rojo'] == 3

# 23 · Todos inválidos → porcentaje no calculable (NaN), sin error
todos_mal = aplicar_regla(validar_entradas(pd.DataFrame({'evaluacion_id': [1, 2], 'fecha': ['2026-08-28'] * 2, 'modulo': [1, 1], 'hojas_inf_pct': ['x', -1]})))
i0 = calcular_indicadores(todos_mal)
assert i0.loc[0, 'Evaluables'] == 0 and np.isnan(i0.loc[0, 'Pct_Ambar_o_Rojo'])

# 24 · Falta una columna obligatoria → error claro
try:
    validar_entradas(pd.DataFrame({'evaluacion_id': [1]})); raise AssertionError('debió fallar')
except ValueError as e:
    assert 'Faltan columnas obligatorias' in str(e)

# 25-27 · Semáforo por módulo con el fixture: ventana de 14 días hasta 2026-08-28, mínimo de evaluaciones configurable
sf = semaforo_modulos(ts, fecha_corte='2026-08-28', ventana_dias=14, min_eval=2)
fila4 = sf[sf['modulo'] == 4].iloc[0]
assert fila4['evaluaciones_en_ventana'] == 2 and fila4['estado_modulo'] == 'Ámbar' and abs(fila4['hojas_inf_media'] - 10) < 1e-9   # media (5+15)/2 = 10 → Ámbar (borde)
sf2 = semaforo_modulos(ts, fecha_corte='2026-08-28', ventana_dias=14, min_eval=5)
assert (sf2['estado_modulo'] == 'Datos insuficientes').all()
sf3 = semaforo_modulos(ts, fecha_corte='2026-08-28', ventana_dias=3, min_eval=1)    # ventana corta: 08-26 a 08-28
assert sf3[sf3['modulo'] == 5].iloc[0]['estado_modulo'] == 'Datos insuficientes'     # el dato del 08-20 queda fuera
print('✓ Pruebas de integración superadas (27 comprobaciones)')

In [ ]:
# 28-30 · Consistencia sobre el dataset real
assert resultado['estado'].notna().all()
assert (resultado.loc[~resultado['evaluable'], 'estado'] == 'No evaluable').all()
assert (resultado.loc[resultado['evaluable'] & (resultado['hojas_inf_pct'] >= UMBRAL_ROJO), 'estado'] == 'Rojo').all()
assert (resultado.loc[resultado['evaluable'] & (resultado['hojas_inf_pct'] < UMBRAL_AMBAR), 'estado'] == 'Verde').all()
assert int((resultado['estado'] == 'Rojo').sum()) == 147 and int((resultado['estado'] == 'Ámbar').sum()) == 326   # 473 - 147
assert len(revision) == int(resultado[resultado['estado'].isin(['Ámbar', 'Rojo']) & (resultado['fecha'] >= inicio)].shape[0])
assert set(semaforo['estado_modulo']) <= {'Verde', 'Ámbar', 'Rojo', 'Datos insuficientes'}
assert (semaforo.loc[semaforo['observacion'] != '', 'evaluaciones_rojas'] > 0).all()
assert semaforo.loc[semaforo['modulo'] == 4, 'observacion'].iloc[0].startswith('Revisar focos')   # módulo 4: Verde por promedio, pero con evaluaciones en Rojo
print('✓ OK: pruebas de la regla, de integración y de consistencia sobre datos reales superadas.')

## PASO 10 · Sensibilidad a los umbrales
Como los umbrales son provisionales, mostramos cuánto cambia el resultado si se mueven. **Esto es lo que se discutirá con el especialista.**

In [ ]:
filas = []
for a, r_ in [(5, 20), (10, 30), (15, 40)]:
    rr = aplicar_regla(trabajo, a, r_); sm = semaforo_modulos(rr, umbral_ambar=a, umbral_rojo=r_)
    ind = calcular_indicadores(rr, a, r_).iloc[0]
    filas.append({'ámbar': a, 'rojo': r_, 'pct_ámbar_o_rojo_global': round(ind['Pct_Ambar_o_Rojo'], 1), 'pct_rojo_global': round(ind['Pct_Rojo'], 1),
                  'módulos_en_rojo': int((sm['estado_modulo'] == 'Rojo').sum()), 'módulos_en_ámbar': int((sm['estado_modulo'] == 'Ámbar').sum()),
                  'módulos_en_verde': int((sm['estado_modulo'] == 'Verde').sum())})
sens = pd.DataFrame(filas); display(sens)

## PASO 11 · Exportar evidencias

In [ ]:
r1 = SALIDA / 'semana11_evaluaciones_clasificadas.csv'
r2 = SALIDA / 'semana11_revision_ventana.csv'
r3 = SALIDA / 'semana11_indicadores.csv'
r4 = SALIDA / 'semana11_semaforo_modulos.csv'
r5 = SALIDA / 'semana11_sensibilidad_umbrales.csv'
resultado.assign(fecha=resultado['fecha'].dt.strftime('%Y-%m-%d'))[['evaluacion_id', 'fecha', 'modulo', 'turno', 'valvula', 'hojas_inf_pct', 'evaluable', 'motivo', 'estado']].to_csv(r1, index=False, encoding='utf-8-sig')
revision.assign(fecha=revision['fecha'].dt.strftime('%Y-%m-%d')).to_csv(r2, index=False, encoding='utf-8-sig')
indicadores.to_csv(r3, index=False, encoding='utf-8-sig'); semaforo.to_csv(r4, index=False, encoding='utf-8-sig'); sens.to_csv(r5, index=False, encoding='utf-8-sig')
for ruta in [r1, r2, r3, r4, r5]:
    assert ruta.exists(); print(f'Generado: {ruta.name} ({ruta.stat().st_size} bytes)')
assert len(pd.read_csv(r1)) == len(resultado) and len(pd.read_csv(r2)) == len(revision) and len(pd.read_csv(r3)) == 1 and len(pd.read_csv(r4)) == 6
print('OK: exportaciones verificadas.')

## FICHA TÉCNICA DEL COMPONENTE
| Campo | Contenido |
|---|---|
| **Nombre** | Semáforo de infestación por arañita roja por módulo (v1) |
| **Necesidad / decisión apoyada** | Priorizar qué módulos revisar o intervenir cada semana según la infestación observada |
| **Entradas** | `evaluacion_id`, `fecha`, `modulo`, `hojas_inf_pct` (del dataset analítico v1); parámetros `UMBRAL_AMBAR`, `UMBRAL_ROJO`, `VENTANA_DIAS`, `MIN_EVAL`, `FECHA_CORTE` |
| **Procesamiento** | Validación de campos → regla determinística por evaluación → promedio de la ventana reciente por módulo → clasificación (más aviso de focos en Rojo) → indicadores y tabla de revisión |
| **Salidas** | Estado por evaluación; indicadores globales; semáforo por módulo; tabla de revisión; sensibilidad a umbrales |
| **Técnica** | Reglas explícitas y estadística descriptiva en pandas. No se entrena ningún modelo |
| **Pruebas** | Más de 25 comprobaciones: normales, límites (10 y 30), extremos (0 y 100), inválidos, umbral configurable, datos sucios, todos inválidos, columna faltante y ventana |
| **Evidencia** | Este notebook, 5 CSV de salida y la salida de las pruebas |
| **Limitaciones** | Umbrales provisionales; un solo ciclo de 6 meses; módulo 1 con pocas evaluaciones; Tbase y K no validados en campo; el semáforo describe lo observado, **no predice** ni demuestra causas |

## JUSTIFICACIÓN DE LA TÉCNICA (5 criterios)  
**Por qué reglas determinísticas y no un modelo de Machine Learning**
1. **Pertinencia al problema.** El especialista decide con niveles de infestación observada; una regla con umbrales explícitos responde exactamente esa decisión.
2. **Calidad y cantidad de datos.** Hay 1,462 evaluaciones, pero de un solo ciclo y con solo 50 fechas de clima; el GDA mostró una relación débil con la infestación (ρ = 0.25, p = 0.08). Con esa evidencia un modelo predictivo no sería defendible.
3. **Interpretabilidad.** Cada color se explica con una comparación contra un umbral; el técnico puede auditarlo sin conocimientos de modelos.
4. **Riesgo de error.** Los errores de una regla son visibles y ajustables (cambiando un parámetro). Un modelo mal validado daría falsa seguridad.
5. **Viabilidad.** Es implementable en Colab con pandas, se prueba con `assert` y se puede reproducir en la sustentación.

**Coherencia ficha–implementación:** la ficha y el código describen el mismo componente (mismas entradas, mismos parámetros, mismas salidas).

**Extensión posible (solo si los datos lo respaldan):** cuando existan más campañas y umbrales validados, se podría evaluar un modelo supervisado que anticipe el paso a Ámbar/Rojo usando GDA, humedad e infestación previa, comparándolo contra esta regla como línea base. Hoy **no** se justifica.

**Mensaje honesto para la sustentación:** el sistema no “predice la plaga con grados-día”; **vigila la infestación observada**, y usa el grado-día como contexto térmico mientras se valida si puede anticipar riesgo.

In [ ]:
zip_path = shutil.make_archive('semana11_evidencias_AgroDataIC', 'zip', root_dir=SALIDA)
print('Paquete creado:', zip_path)
try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Fuera de Colab, el ZIP queda guardado en la carpeta de ejecución.')

# CHECKLIST FINAL
- [x] Decisión apoyada, entradas, procesamiento y salidas definidas
- [x] Validación de entradas: los inválidos no se convierten a 0 ni se eliminan
- [x] Regla determinística con parámetros configurables y documentados como provisionales
- [x] Indicadores con denominador solo de evaluables
- [x] Tabla de revisión
- [x] Al menos 3 pruebas, incluyendo caso límite e inválido (más de 25 comprobaciones)
- [x] Ficha técnica y justificación de la técnica
- [x] Evidencias exportadas